<a href="https://colab.research.google.com/github/isnanafis/Data_Preprocessing/blob/main/datapreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ***Data Preprocessing & Wrangling*** **: Dataset** ***Messy Food Waste***
## **Konteks dan Sumber Dataset**


* Platform: Kaggle Community Competition
* Nama Dataset: Messy Food Waste Prediction Dataset
* Tautan: [Kaggle - Messy Food Waste Prediction](https://www.kaggle.com/competitions/messy-food-waste-prediction-dataset)

Dataset ini merupakan data operasional harian dapur komersial yang digunakan untuk memprediksi jumlah limbah makanan (`food_waste_kg`). Dataset ini memiliki beberapa masalah kualitas data, seperti format teks yang tidak seragam, tipe data yang tidak tepat, serta adanya nilai yang hilang (*missing values*).

## 1. Memuat dan Inspeksi Data Awal
Tahap paling awal sebelum melakukan manipulasi apa pun adalah memuat dataset dan menginspeksinya. Kita perlu melihat tipe data, jumlah kolom, dan sampel baris untuk mengidentifikasi keberadaan nilai kosong (*missing values*) atau kolom yang tidak berguna.

**Sintaks:**
* `pd.read_csv()`: Memuat file CSV menjadi DataFrame Pandas.
* `df.info()`: Memeriksa jumlah baris, tipe data, dan deteksi awal *missing values*.
* `df.head()`: Menampilkan 5 baris pertama untuk inspeksi visual.

In [3]:
import pandas as pd
import numpy as np
from sklearn import preprocessing

# 1. Memuat data
df = pd.read_csv('food_waste_dataset.csv')

# 2. Inspeksi struktur dan tipe data
print("Informasi Dataset:")
df.info()

# 3. Inspeksi visual sampel data
print("\nSampel 5 Baris Pertama:")
display(df.head())

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911 entries, 0 to 910
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                911 non-null    int64  
 1   date              911 non-null    object 
 2   meals_served      911 non-null    int64  
 3   kitchen_staff     911 non-null    int64  
 4   temperature_C     911 non-null    float64
 5   humidity_percent  911 non-null    float64
 6   day_of_week       911 non-null    int64  
 7   special_event     911 non-null    int64  
 8   past_waste_kg     911 non-null    float64
 9   staff_experience  747 non-null    object 
 10  waste_category    911 non-null    object 
 11  food_waste_kg     911 non-null    float64
dtypes: float64(4), int64(5), object(3)
memory usage: 85.5+ KB

Sampel 5 Baris Pertama:


,ID,date,meals_served,kitchen_staff,temperature_C,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465
1,1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,NaN,MeAt,51.549053
2,4,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,Beginner,MeAt,53.008323
3,5,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,Beginner,MeAt,48.621527
4,6,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,Intermediate,MEAT,44.156984


**Penjelasan Output:**
Dari output `.info()` dan `.head()` di atas, kita dapat menemukan beberapa temuan penting: ada kolom `ID` yang murni hanya nomor urut acak (tidak memiliki makna statistik), nama kolom memiliki format huruf besar/kecil yang tidak konsisten, dan ada nilai *NaN* pada atribut `staff_experience`. Ini menjadi dasar untuk langkah pembersihan kita selanjutnya.

## 2. Pembersihan Struktur Dasar
Berdasarkan hasil inspeksi di Langkah 1, kolom `ID` dipastikan tidak berguna untuk analisis prediksi, sehingga harus dihapus. Kita juga memastikan tidak ada data yang terekam dua kali untuk mencegah bias.

**Sintaks:**
* `drop(columns=['ID'])`: Menghilangkan kolom identitas.
* `drop_duplicates()`: Menghapus baris yang datanya persis sama/berulang.

In [4]:
# Menghapus kolom yang tidak memiliki nilai analitis
df.drop(columns=['ID'], inplace=True)

# Menghapus observasi duplikat
df.drop_duplicates(inplace=True)

print(f"Dimensi data setelah pembersihan struktur: {df.shape}")

Dimensi data setelah pembersihan struktur: (911, 11)


**Penjelasan Output:**
Output dimensi data (`shape`) kini telah berkurang. Penghapusan kolom `ID` membuat dataset lebih ringan di memori, dan eliminasi duplikat menjamin bahwa model statistik kita nantinya belajar dari observasi yang murni unik.

## 3. Standardisasi Nama Kolom (*Column Naming*)
Untuk memudahkan pemanggilan variabel saat proses *coding*, seluruh nama atribut diseragamkan ke dalam format *snake_case*.

**Sintaks:**
* `df.rename()`: Dipadukan dengan *list comprehension* Python. Kita memanggil `.lower()` untuk mengubah teks menjadi huruf kecil dan `.replace(' ', '_')` untuk mengganti spasi menjadi garis bawah.

In [5]:
# Standardisasi format nama kolom
df = df.rename(columns={col: col.lower().replace(' ', '_') for col in df.columns.values.tolist()})

print("Daftar nama kolom saat ini:")
print(df.columns.tolist())

Daftar nama kolom saat ini:
['date', 'meals_served', 'kitchen_staff', 'temperature_c', 'humidity_percent', 'day_of_week', 'special_event', 'past_waste_kg', 'staff_experience', 'waste_category', 'food_waste_kg']


**Penjelasan Output:**
Daftar kolom yang dicetak kini sepenuhnya konsisten. Kolom yang sebelumnya mungkin ditulis sebagai "User Type" atau "Serial No" kini telah otomatis berubah menjadi `user_type` dan `serial_no`.

## 4. Penyeragaman Format Teks (*String Manipulation*)
Data teks yang diketik manual sangat rentan terhadap kesalahan kapitalisasi (contoh: "MeAt" vs "meat") atau kelebihan spasi.

**Sintaks:**
* `.str.strip()`: Menghilangkan spasi kosong di awal dan akhir teks.
* `.str.lower()`: Memaksa semua karakter menjadi huruf kecil agar seragam.

In [6]:
# Membersihkan teks pada kolom kategorikal
df['waste_category'] = df['waste_category'].str.strip().str.lower()
df['staff_experience'] = df['staff_experience'].str.strip().str.lower()
df.head()

,date,meals_served,kitchen_staff,temperature_c,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg
0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465
1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,NaN,meat,51.549053
2,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,beginner,meat,53.008323
3,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,beginner,meat,48.621527
4,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,intermediate,meat,44.156984


**Penjelasan Output:**
Langkah ini menyatukan kategori yang sebelumnya terpecah hanya karena salah ketik. Sistem kini membaca "MeAt" dan " MEAT " sebagai satu kategori tunggal yang sama, yaitu "meat".

## 5. Penyesuaian Tipe Data (*Typecasting*)
Sistem tidak dapat menghitung angka jika format datanya masih berupa teks (*string*). Kolom kuantitatif dan waktu harus dikonversi ke tipe data yang sesuai.

**Sintaks:**
* `pd.to_datetime()`: Mengonversi *string* tanggal menjadi objek *Datetime* fungsional.
* `str.replace(r'\D+', '')`: Ekspresi reguler (*Regex*) untuk mengekstrak hanya digit angka, membuang simbol aneh seperti "150*" menjadi "150".
* `pd.to_numeric()`: Mengubah hasil ekstraksi menjadi angka integer/float.

In [7]:
# Konversi kolom tanggal
df['date'] = pd.to_datetime(df['date'])

# Membersihkan karakter non-angka dan mengubah menjadi tipe numerik
df['meals_served'] = pd.to_numeric(df['meals_served'].astype(str).str.replace(r'\D+', '', regex=True))

print("Tipe Data Saat Ini:")
print(df.dtypes)

Tipe Data Saat Ini:
date                datetime64[ns]
meals_served                 int64
kitchen_staff                int64
temperature_c              float64
humidity_percent           float64
day_of_week                  int64
special_event                int64
past_waste_kg              float64
staff_experience            object
waste_category              object
food_waste_kg              float64
dtype: object


**Penjelasan Output:**
Output `dtypes` memvalidasi keberhasilan konversi. Kolom `date` kini dikenali sebagai `datetime64`, dan `meals_served` kini bertipe numerik yang siap dikenakan operasi penjumlahan atau rata-rata.

## 6. Imputasi dan Penanganan Nilai Kosong (*Missing Values*)
Kebocoran data diatasi menggunakan metode yang disesuaikan dengan jenis variabelnya.

**Sintaks:**
* `dropna(subset=['date'])`: Baris dihapus jika tanggalnya hilang (data *time-series* tidak valid tanpa waktu).
* `fillna(mean)`: Mengisi kekosongan `temperature_c` dengan rata-rata dari seluruh data suhu.
* `.ffill()` & `.bfill()`: Menggunakan *Forward Fill* (mengisi dengan data dari baris sebelumnya) dan *Backward Fill* untuk variabel kategori operasional.

In [8]:
# 1. Hapus observasi tanpa tanggal
df.dropna(subset=['date'], inplace=True)

# 2. Imputasi rata-rata pada fitur suhu
df['temperature_c'] = df['temperature_c'].fillna(value=np.round(df['temperature_c'].mean(), decimals=2))

# 3. Imputasi Forward/Backward Fill pada fitur kategorikal (pengalaman staf)
df['staff_experience'] = df['staff_experience'].ffill()
df['staff_experience'] = df['staff_experience'].bfill()

print("Jumlah missing values tersisa:\n", df.isnull().sum())

Jumlah missing values tersisa:
 date                0
meals_served        0
kitchen_staff       0
temperature_c       0
humidity_percent    0
day_of_week         0
special_event       0
past_waste_kg       0
staff_experience    0
waste_category      0
food_waste_kg       0
dtype: int64


/tmp/ipykernel_2370/2198303362.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['temperature_c'].fillna(value=np.round(df['temperature_c'].mean(), decimals=2), inplace=True)


**Penjelasan Output:**
Output mengonfirmasi bahwa seluruh nilai *null* kini adalah 0. Distribusi statistik suhu tetap aman karena diisi dengan rata-rata, sedangkan urutan data operasional staf terselamatkan menggunakan teknik *fill*.

## 7. Ekstraksi Fitur (*Feature Engineering*)
Kita memecah informasi temporal menjadi variabel baru yang berdiri sendiri untuk menangkap pola musiman atau pola hari kerja.

**Sintaks:**
* `.dt.month`: Mengekstrak indeks bulan (1-12).
* `.dt.dayofweek`: Mengekstrak indeks hari dalam seminggu (0-6).

In [9]:
# Membuat variabel turunan dari kolom tanggal
df['operational_month'] = df['date'].dt.month
df['operational_day_of_week'] = df['date'].dt.dayofweek
df.head()

,date,meals_served,kitchen_staff,temperature_c,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,waste_category,food_waste_kg,operational_month,operational_day_of_week
0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,dairy,28.946465,12,0
1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,intermediate,meat,51.549053,11,1
2,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,beginner,meat,53.008323,2,1
3,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,beginner,meat,48.621527,3,6
4,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,intermediate,meat,44.156984,7,0


**Penjelasan Output:**
Dua kolom prediktor baru telah terbentuk. Ini memungkinkan model regresi spasial atau temporal kita nanti untuk belajar, misalnya: "apakah hari Sabtu menghasilkan lebih banyak limbah dibanding hari Senin?".

## 8. Pengodean Variabel Kategorikal (*Encoding*)
Teks harus diubah menjadi matriks angka agar dapat dibaca oleh algoritma komputasi.

**Sintaks:**
* `pd.get_dummies()`: Melakukan *One-Hot Encoding* memecah variabel jenis limbah menjadi beberapa kolom biner (0/1).
* `.map()`: Melakukan *Label Mapping* dengan basis *dictionary* untuk merepresentasikan tingkatan pengalaman staf sebagai bilangan ordinal (0, 1, 2).

In [10]:
# 1. One-Hot Encoding (Untuk kategori tanpa hierarki)
df = pd.get_dummies(df, columns=['waste_category'])

# 2. Label Mapping (Untuk kategori yang memiliki tingkatan/hierarki)
exp_map = {'beginner': 0, 'intermediate': 1, 'expert': 2}
df['encoded_experience'] = df['staff_experience'].map(exp_map)
df.head()

,date,meals_served,kitchen_staff,temperature_c,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,food_waste_kg,operational_month,operational_day_of_week,waste_category_dairy,waste_category_grains,waste_category_meat,waste_category_vegetables,encoded_experience
0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,28.946465,12,0,True,False,False,False,1
1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,intermediate,51.549053,11,1,False,False,True,False,1
2,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,beginner,53.008323,2,1,False,False,True,False,0
3,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,beginner,48.621527,3,6,False,False,True,False,0
4,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,intermediate,44.156984,7,0,False,False,True,False,1


**Penjelasan Output:**
Fitur berbasis teks telah berhasil dienkode. Kategori limbah direpresentasikan secara boolean/biner, sedangkan pengalaman staf diproyeksikan sebagai angka berurut, sehingga mesin dapat menangkap logika bahwa `2 (Expert)` > `0 (Beginner)`.

## 9. Normalisasi Skala Data Numerik (*Scaling*)
Langkah penutup adalah menormalisasi angka agar fitur berskala besar (seperti porsi makan) tidak mendominasi perhitungan gradien dibandingkan fitur berskala kecil (suhu).

**Sintaks:**
* `MinMaxScaler()`: Menyelaraskan seluruh nilai interval suhu ke dalam proporsi rasio 0.0 hingga 1.0.
* `RobustScaler()`: Mentransformasi porsi makan menggunakan metode yang kebal terhadap pencilan (*outliers*), sehingga hari dengan lonjakan pengunjung ekstrem tidak merusak distribusi model.

In [11]:
# Normalisasi Suhu
min_max_scaler = preprocessing.MinMaxScaler()
df['temperature_c_scaled'] = min_max_scaler.fit_transform(df['temperature_c'].values.reshape(-1, 1))

# Normalisasi Porsi Makan
robust_scaler = preprocessing.RobustScaler()
df['meals_served_scaled'] = robust_scaler.fit_transform(df['meals_served'].values.reshape(-1, 1))

print(f"Bentuk Dimensi Akhir Dataset: {df.shape}")
display(df.head())

Bentuk Dimensi Akhir Dataset: (911, 19)


,date,meals_served,kitchen_staff,temperature_c,humidity_percent,day_of_week,special_event,past_waste_kg,staff_experience,food_waste_kg,operational_month,operational_day_of_week,waste_category_dairy,waste_category_grains,waste_category_meat,waste_category_vegetables,encoded_experience,temperature_c_scaled,meals_served_scaled
0,2022-12-19,196,13,27.887273,45.362854,0,0,7.740587,intermediate,28.946465,12,0,True,False,False,False,1,0.543673,-0.561224
1,2023-11-21,244,15,10.317872,64.430475,1,0,42.311779,intermediate,51.549053,11,1,False,False,True,False,1,0.294009,-0.316327
2,2022-02-01,148,16,27.714300,69.046113,1,0,41.184305,beginner,53.008323,2,1,False,False,True,False,0,0.541215,-0.806122
3,2023-03-19,157,19,19.173902,46.292823,6,0,41.543492,beginner,48.621527,3,6,False,False,True,False,0,0.419855,-0.760204
4,2022-07-18,297,10,26.375233,79.741064,0,0,26.525097,intermediate,44.156984,7,0,False,False,True,False,1,0.522187,-0.045918


**Penjelasan Output:**
Tabel yang ditampilkan di atas adalah hasil akhir dari seluruh proses pembersihan data. Sekarang, sudah tidak ada lagi data yang kosong, format teksnya sudah seragam, informasi waktu sudah diekstrak, dan skala angkanya sudah seimbang. Singkatnya, dataset *Food Waste* ini sudah benar-benar rapi dan siap digunakan untuk tahap analisis atau pemodelan statistik selanjutnya.